### Load packages

In [80]:
import pandas as pd
import os
import numpy as np

### Load data files

In [81]:
# Set parameters for data files

data_directory = r"..\data\unified_csvs"
output_directory = r"..\data\merged_csvs"

vct_file = "vct_unified_prepped"
vct_path = os.path.join(data_directory, vct_file + ".csv")

output_file = "algae_merged"
output_path = os.path.join(output_directory, output_file + ".csv")

merge_config = {
    "usgs_prepped": {
        "file": "usgs_prepped.csv",
        "left_on": ["vct_report_date"],
        "right_on": ["usgs_report_date"],
        "how": "inner"
    },
    "noaa_weather_avg": {
        "file": "noaa_weather_avg.csv",
        "left_on": ["vct_report_date"],
        "right_on": ["noaa_DATE"],
        "how": "inner"
    },
    "vt_dec_unified_prepped": {
        "file": "vt_dec_unified_prepped.csv",
        "left_on": ["vct_report_date", "vct_region"],
        "right_on": ["dec_vct_report_date", "dec_region"],
        "how": "inner"
    }
}

In [82]:
# Load VCT data file

vct_df = pd.read_csv(vct_path)

# Add vct_ prefix to feature columns if it isn't already there
vct_df.rename(
    columns=lambda col: (
        col
        if col.startswith("vct_")
        else f"vct_{col}"
    ),
    inplace=True
)

# Make sure the report_date column is a date field
vct_df["vct_report_date"] = pd.to_datetime(
    vct_df["vct_report_date"]
)

vct_df.head()

,vct_region,vct_report_date,vct_year,vct_latitude,vct_longitude,vct_water_temp,vct_water_surface,vct_anabaena,vct_aphanizomenon,vct_microcystin,...,vct_oscillatoria_7days,vct_target_bloom_7days,vct_water_temp_14days,vct_water_surface_14days,vct_target_bloom_intensity_14days,vct_anabaena_14days,vct_aphanizomenon_14days,vct_microcystin_14days,vct_oscillatoria_14days,vct_target_bloom_14days
0,Inland Sea,2012-07-17,2012,44.801952,-73.257075,NaN,NaN,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Inland Sea,2012-07-24,2012,44.801952,-73.257075,NaN,NaN,0,0,0,...,0.0,0.0,NaN,NaN,1.000000,0.0,0.0,0.0,0.0,0.0
2,Inland Sea,2013-06-16,2013,44.801952,-73.257075,60.400,1.666667,0,0,0,...,0.0,0.0,NaN,NaN,1.000000,0.0,0.0,0.0,0.0,0.0
3,Inland Sea,2013-06-17,2013,44.801952,-73.257075,61.075,1.600000,0,0,0,...,0.0,0.0,60.4000,1.666667,1.027778,0.0,0.0,0.0,0.0,0.0
4,Inland Sea,2013-06-18,2013,44.801952,-73.257075,62.000,1.000000,0,0,0,...,0.0,0.0,60.7375,1.633333,1.033333,0.0,0.0,0.0,0.0,0.0


In [83]:
# Load component data files

component_dfs = {}

for file, merge_info in merge_config.items():

    print(f"Loading {file}")
    print(merge_info)

    path = os.path.join(data_directory, merge_info["file"])

    component_dfs[file] = pd.read_csv(path)

Loading usgs_prepped
{'file': 'usgs_prepped.csv', 'left_on': ['vct_report_date'], 'right_on': ['usgs_report_date'], 'how': 'inner'}
Loading noaa_weather_avg
{'file': 'noaa_weather_avg.csv', 'left_on': ['vct_report_date'], 'right_on': ['noaa_DATE'], 'how': 'inner'}
Loading vt_dec_unified_prepped
{'file': 'vt_dec_unified_prepped.csv', 'left_on': ['vct_report_date', 'vct_region'], 'right_on': ['dec_vct_report_date', 'dec_region'], 'how': 'inner'}


### QC component data files, reformat and rename columns

In [84]:
# noaa weather data

# Add noaa_ prefix to feature columns if it isn't already there
component_dfs["noaa_weather_avg"].rename(
    columns=lambda col: (
        col
        # if col == "DATE" or col.startswith("noaa_")
        if col.startswith("noaa_")
        else f"noaa_{col}"
    ),
    inplace=True
)

# Make sure the DATE column is a date field
component_dfs["noaa_weather_avg"]["noaa_DATE"] = pd.to_datetime(
    component_dfs["noaa_weather_avg"]["noaa_DATE"]
)

component_dfs["noaa_weather_avg"].head()

,noaa_DATE,noaa_precipitation,noaa_air_temp_max,noaa_air_temp_min,noaa_wind_speed_mean,noaa_wind_speed_2_min,noaa_snow_depth,noaa_wind_direction_2_min,noaa_snowfall,noaa_wind_speed_5_min,noaa_wind_direction_5_min,noaa_air_temp_mean
0,2012-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-01-02,0.50,7.200,-1.1,3.40,10.300000,0.0,180.0,0.000000,14.300000,190.000000,NaN
2,2012-01-03,1.50,6.950,-3.6,4.05,10.300000,0.0,255.0,2.500000,13.650000,265.000000,NaN
3,2012-01-04,1.00,2.600,-7.6,4.50,9.833333,0.0,280.0,3.333333,12.833333,286.666667,NaN
4,2012-01-05,0.75,0.425,-10.0,4.45,9.600000,0.0,257.5,2.500000,12.525000,260.000000,NaN


In [85]:
# USGS data

# Add usgs_ prefix to feature columns if it isn't already there
component_dfs["usgs_prepped"].rename(
    columns=lambda col: (
        col
        if col.startswith("usgs_")
        else f"usgs_{col}"
    ),
    inplace=True
)

# Make sure the report_date column is a date field
component_dfs["usgs_prepped"]["usgs_report_date"] = pd.to_datetime(
    component_dfs["usgs_prepped"]["usgs_report_date"]
)

component_dfs["usgs_prepped"].head()

,usgs_site,usgs_report_date,usgs_latitude,usgs_longitude,usgs_water_temp_max,usgs_water_temp_min,usgs_water_temp_mean,usgs_conductivity_max,usgs_conductivity_min,usgs_conductivity_mean,...,usgs_water_temp_mean_7days,usgs_conductivity_max_7days,usgs_conductivity_min_7days,usgs_conductivity_mean_7days,usgs_water_temp_max_14days,usgs_water_temp_min_14days,usgs_water_temp_mean_14days,usgs_conductivity_max_14days,usgs_conductivity_min_14days,usgs_conductivity_mean_14days
0,4294500,2014-09-30,44.47616,-73.221517,18.7,17.2,17.9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4294500,2014-10-01,44.47616,-73.221517,17.2,15.8,16.5,179.0,170.0,173.0,...,17.900000,NaN,NaN,NaN,18.700000,17.200000,17.900000,NaN,NaN,NaN
2,4294500,2014-10-02,44.47616,-73.221517,17.0,15.8,16.4,177.0,170.0,174.0,...,17.200000,179.000000,170.0,173.0,17.950000,16.500000,17.200000,179.000000,170.0,173.0
3,4294500,2014-10-03,44.47616,-73.221517,17.3,16.5,16.9,179.0,173.0,175.0,...,16.933333,178.000000,170.0,173.5,17.633333,16.266667,16.933333,178.000000,170.0,173.5
4,4294500,2014-10-04,44.47616,-73.221517,17.1,16.4,16.8,191.0,161.0,176.0,...,16.925000,178.333333,171.0,174.0,17.550000,16.325000,16.925000,178.333333,171.0,174.0


In [86]:
# DEC data

# Add dec_ prefix to feature columns if it isn't already there
component_dfs["vt_dec_unified_prepped"].rename(
    columns=lambda col: (
        col
        if col.startswith("dec_")
        else f"dec_{col}"
    ),
    inplace=True
)

# Make sure the report_date column is a date field
component_dfs["vt_dec_unified_prepped"]["dec_vct_report_date"] = pd.to_datetime(
    component_dfs["vt_dec_unified_prepped"]["dec_vct_report_date"]
)

component_dfs["vt_dec_unified_prepped"].head()

,dec_region,dec_vct_report_date,dec_total nitrogen,dec_total phosphorus,dec_dissolved phosphorus,dec_chlorophyll-a,dec_secchi depth,dec_temperature
0,Main Lake Central,2012-07-01,NaN,NaN,NaN,NaN,NaN,NaN
1,Main Lake Central,2012-07-02,NaN,NaN,NaN,NaN,NaN,NaN
2,Main Lake Central,2012-07-03,NaN,NaN,NaN,NaN,NaN,NaN
3,Main Lake Central,2012-07-07,NaN,NaN,NaN,NaN,NaN,NaN
4,Main Lake Central,2012-07-08,NaN,NaN,NaN,NaN,NaN,NaN


### Merge component data files

In [87]:
# Merge components using merge_config info

algae_merged_df = vct_df.copy()

for file, merge_info in merge_config.items():
    
    print(f"Merging {file}")
    
    right_df = component_dfs[file]

    algae_merged_df = algae_merged_df.merge(
        right_df,
        left_on=merge_info.get("left_on"),
        right_on=merge_info.get("right_on"),
        # on=merge_info.get("on"),
        how=merge_info.get("how", "left")
    )

print("Rows:", len(algae_merged_df))
print("Columns:", len(algae_merged_df.columns))
print(algae_merged_df.columns)

Merging usgs_prepped
Merging noaa_weather_avg
Merging vt_dec_unified_prepped
Rows: 2919
Columns: 72
Index(['vct_region', 'vct_report_date', 'vct_year', 'vct_latitude',
       'vct_longitude', 'vct_water_temp', 'vct_water_surface', 'vct_anabaena',
       'vct_aphanizomenon', 'vct_microcystin', 'vct_oscillatoria',
       'vct_target_bloom', 'vct_target_bloom_intensity', 'vct_day_of_year',
       'vct_water_temp_7days', 'vct_water_surface_7days',
       'vct_target_bloom_intensity_7days', 'vct_anabaena_7days',
       'vct_aphanizomenon_7days', 'vct_microcystin_7days',
       'vct_oscillatoria_7days', 'vct_target_bloom_7days',
       'vct_water_temp_14days', 'vct_water_surface_14days',
       'vct_target_bloom_intensity_14days', 'vct_anabaena_14days',
       'vct_aphanizomenon_14days', 'vct_microcystin_14days',
       'vct_oscillatoria_14days', 'vct_target_bloom_14days', 'usgs_site',
       'usgs_report_date', 'usgs_latitude', 'usgs_longitude',
       'usgs_water_temp_max', 'usgs_water_tem

### Clean up redundant columns

In [88]:
# Drop duplicate columns

dupe_cols = ["vct_water_surface", "vct_water_temp",
             "vct_anabaena", "vct_aphanizomenon", "vct_microcystin", "vct_oscillatoria",
             "usgs_site", "usgs_report_date", "usgs_latitude", "usgs_longitude",
             "usgs_water_temp_max", "usgs_water_temp_min", "usgs_water_temp_mean", 
             "usgs_conductivity_max", "usgs_conductivity_min", "usgs_conductivity_mean",
             "noaa_DATE",
             "dec_region", "dec_vct_report_date"]

algae_cleancols_df = algae_merged_df.copy()

algae_cleancols_df = algae_cleancols_df.drop(columns=dupe_cols)

algae_cleancols_df

,vct_region,vct_report_date,vct_year,vct_latitude,vct_longitude,vct_target_bloom,vct_target_bloom_intensity,vct_day_of_year,vct_water_temp_7days,vct_water_surface_7days,...,noaa_snowfall,noaa_wind_speed_5_min,noaa_wind_direction_5_min,noaa_air_temp_mean,dec_total nitrogen,dec_total phosphorus,dec_dissolved phosphorus,dec_chlorophyll-a,dec_secchi depth,dec_temperature
0,Main Lake Central,2014-09-30,2014,44.484736,-73.288936,0,1.0000,273,68.240,1.516667,...,0.0,9.035714,211.428571,15.228571,0.340000,12.500000,5.000000,3.830,5.400000,NaN
1,Main Lake Central,2014-10-02,2014,44.484736,-73.288936,1,3.0000,275,68.240,1.220000,...,0.0,8.842857,233.571429,15.328571,0.340000,12.500000,5.000000,3.830,5.400000,NaN
2,Main Lake Central,2015-06-14,2015,44.484736,-73.288936,0,1.1250,165,70.175,1.125000,...,0.0,11.521429,254.285714,16.521429,0.400000,69.600000,13.000000,2.300,5.000000,12.0
3,Main Lake Central,2015-06-15,2015,44.484736,-73.288936,0,1.0000,166,68.090,1.200000,...,0.0,10.914286,250.000000,16.992857,0.400000,69.600000,13.000000,2.300,5.000000,12.0
4,Main Lake Central,2015-06-16,2015,44.484736,-73.288936,0,1.0000,167,65.690,1.400000,...,0.0,11.007143,252.142857,17.557143,0.405000,39.700000,9.650000,1.815,4.050000,14.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,St. Albans Bay,2022-10-05,2022,44.788956,-73.168235,1,2.7500,278,70.000,1.238095,...,0.0,10.514286,267.857143,12.178571,0.536667,33.916667,23.716667,23.200,2.333333,NaN
2915,St. Albans Bay,2022-10-06,2022,44.788956,-73.168235,1,1.6875,279,70.000,1.238095,...,0.0,9.842857,275.000000,11.928571,0.500000,30.600000,29.650000,19.700,2.450000,NaN
2916,St. Albans Bay,2022-10-11,2022,44.788956,-73.168235,0,1.5000,284,NaN,1.285714,...,0.0,8.828571,277.142857,10.885714,0.500000,30.600000,29.650000,19.700,2.450000,NaN
2917,St. Albans Bay,2022-10-13,2022,44.788956,-73.168235,0,1.5000,286,NaN,1.357143,...,0.0,8.735714,265.714286,10.350000,0.420000,28.300000,14.500000,17.600,2.500000,NaN


### Output merged file

In [89]:
# Reorder columns so region & date are first, and target columns are last

first_cols = [c for c in ["vct_region", "vct_report_date", "vct_year", "vct_day_of_year"] if c in algae_cleancols_df.columns]
last_cols = [c for c in ["vct_target_bloom", "vct_target_bloom_intensity"] if c in algae_cleancols_df.columns]

middle_cols = [
    col for col in algae_cleancols_df.columns
    if col not in first_cols + last_cols
]

algae_colorder_df = algae_cleancols_df[first_cols + middle_cols + last_cols]

In [90]:
algae_colorder_df.columns

Index(['vct_region', 'vct_report_date', 'vct_year', 'vct_day_of_year',
       'vct_latitude', 'vct_longitude', 'vct_water_temp_7days',
       'vct_water_surface_7days', 'vct_target_bloom_intensity_7days',
       'vct_anabaena_7days', 'vct_aphanizomenon_7days',
       'vct_microcystin_7days', 'vct_oscillatoria_7days',
       'vct_target_bloom_7days', 'vct_water_temp_14days',
       'vct_water_surface_14days', 'vct_target_bloom_intensity_14days',
       'vct_anabaena_14days', 'vct_aphanizomenon_14days',
       'vct_microcystin_14days', 'vct_oscillatoria_14days',
       'vct_target_bloom_14days', 'usgs_water_temp_max_7days',
       'usgs_water_temp_min_7days', 'usgs_water_temp_mean_7days',
       'usgs_conductivity_max_7days', 'usgs_conductivity_min_7days',
       'usgs_conductivity_mean_7days', 'usgs_water_temp_max_14days',
       'usgs_water_temp_min_14days', 'usgs_water_temp_mean_14days',
       'usgs_conductivity_max_14days', 'usgs_conductivity_min_14days',
       'usgs_conductivity_m

In [91]:
# Output merged data file

algae_colorder_df.to_csv(output_path, index=False)